# Support Vector Machine (SVM)


## Locate the project and import functions

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
candidates_for_root = [Path.cwd(), Path.cwd() / 'FallDetection_Software', Path.cwd().parent / 'FallDetection_Software']
ROOT = next((p for p in candidates_for_root if (p / 'train_svm.py').exists() and (p / 'core.py').exists()), None)
assert ROOT is not None, 'Open this notebook inside your FallDetection_Software folder.'
sys.path.insert(0, str(ROOT))
from core import FEATURES, POLICY
from train_svm import PLAN, prepare_development, fit_candidates, freeze_svm, evaluate_models
from svm_model import load_svm, predict_svm
print('SVM project found. Features:', len(FEATURES))


SVM project found. Features: 12


## Read the fixed plan

In [2]:
print(json.dumps(PLAN, indent=2))
print('Feature names:', ', '.join(FEATURES))

{
  "algorithm": "SVC",
  "kernel": "rbf",
  "scaler": "StandardScaler fit on training subjects only",
  "C_candidates": [
    0.1,
    1.0,
    10.0
  ],
  "gamma_candidates": [
    "scale",
    0.01,
    0.1
  ],
  "threshold_candidates": [
    0.35,
    0.5,
    0.65
  ],
  "class_weight": "balanced",
  "selection": "Validation F2, then precision, smaller C, earlier gamma candidate, higher score threshold.",
  "split": {
    "train": [
      1,
      2,
      3
    ],
    "validation": [
      4
    ],
    "reused_test": [
      5,
      6
    ]
  },
  "score_transform": "sigmoid(decision_function); display score only, NOT calibrated probability",
  "comparison_status": "Exploratory comparison on previously inspected subjects 5-6. Not a new independent final test."
}
Feature names: acc_mean, acc_std, acc_min, acc_max, acc_range, gyro_mean, gyro_std, gyro_max, tilt_change, acc_last, gyro_last, acc_diff_max


## Load training and validation windows

In [3]:
training, validation = prepare_development(ROOT)
xtrain, ytrain = training[2:]
xval, yval = validation[2:]
print('Training features:', xtrain.shape, 'fall windows:', int(ytrain.sum()))
print('Validation features:', xval.shape, 'fall windows:', int(yval.sum()))


Training: 3237 windows; validation: 1147 windows.
Training features: (3237, 12) fall windows: 131
Validation features: (1147, 12) fall windows: 30


## Scale features, train SVMs and select settings


In [4]:
scaler, clf, threshold, candidates = fit_candidates(training, validation)
assert int(scaler.n_samples_seen_) == len(xtrain)
np.testing.assert_allclose(scaler.mean_, np.asarray(xtrain, dtype=np.float64).mean(axis=0))
print('Selected C:', clf.C, 'gamma:', clf.gamma, 'threshold:', threshold)
print('Support vectors:', len(clf.support_), 'scaler training windows:', scaler.n_samples_seen_)


Fitted SVM: C=0.1, gamma=scale, support vectors=1385
Fitted SVM: C=0.1, gamma=0.01, support vectors=1643
Fitted SVM: C=0.1, gamma=0.1, support vectors=1344
Fitted SVM: C=1.0, gamma=scale, support vectors=843
Fitted SVM: C=1.0, gamma=0.01, support vectors=1356
Fitted SVM: C=1.0, gamma=0.1, support vectors=814
Fitted SVM: C=10.0, gamma=scale, support vectors=559
Fitted SVM: C=10.0, gamma=0.01, support vectors=990
Fitted SVM: C=10.0, gamma=0.1, support vectors=550
Selected C: 10.0 gamma: 0.1 threshold: 0.35
Support vectors: 550 scaler training windows: 3237.0


## Freeze the model

In [5]:
model = freeze_svm(scaler, clf, threshold, candidates, ROOT)
print('Saved model/trained_svm.json')
print(model['score_note'])


Frozen selected SVM: C=10.0, gamma=0.1, threshold=0.35
Saved model/trained_svm.json
Sigmoid of the SVM decision margin, for display only. NOT a calibrated fall probability; scores are not directly comparable across algorithms.


## Evaluate and compare

In [6]:
report = evaluate_models(scaler, clf, model, training, validation, ROOT)
comparison = pd.read_csv(ROOT / 'reports/svm_comparison.csv')
columns = ['method', 'accuracy', 'precision', 'recall', 'f1']
print(comparison.loc[comparison['split'].eq('reused_test'), columns].round(4).to_string(index=False))
print('Maximum margin difference:', report['checks']['maximum_absolute_margin_error'])


Evaluated train: 3237 windows; portable SVM margins match sklearn.
Evaluated validation: 1147 windows; portable SVM margins match sklearn.
Evaluated reused_test: 2111 windows; portable SVM margins match sklearn.
       method  accuracy  precision  recall     f1
decision_tree    0.9588     0.3740  0.8214 0.5140
random_forest    0.9730     0.4921  0.5536 0.5210
          svm    0.8901     0.1423  0.6250 0.2318
Maximum margin difference: 3.5567104816891515e-12


## Understand mistakes

In [7]:
for method, answer in report['splits']['reused_test'].items():
    measured = answer['window_metrics']
    (tn, fp), (fn, tp) = measured['confusion_matrix']
    print(f'{method}: {tp} fall windows detected; {fn} missed; {fp} false-positive windows.')
    print(f"  Balanced accuracy: {measured['balanced_accuracy']:.2%}; F1: {measured['f1']:.3f}")


decision_tree: 46 fall windows detected; 10 missed; 77 false-positive windows.
  Balanced accuracy: 89.20%; F1: 0.514
random_forest: 31 fall windows detected; 25 missed; 32 false-positive windows.
  Balanced accuracy: 76.90%; F1: 0.521
svm: 35 fall windows detected; 21 missed; 211 false-positive windows.
  Balanced accuracy: 76.12%; F1: 0.232


## Predict from CSV inputs

In [8]:
loaded = load_svm(ROOT / 'model/trained_svm.json')
for kind in ['walking', 'fall']:
    frame = pd.read_csv(ROOT / f'examples/{kind}_window.csv')
    answers = predict_svm(loaded, frame)
    print(kind, 'known training excerpt:')
    for answer in answers:
        label = 'Suspected fall' if answer['suspected_fall'] else 'No fall predicted'
        print(f"  {answer['time_s']:.2f} s: {label}; score {answer['fall_score']:.3f}; margin {answer['svm_margin']:.3f}")


walking known training excerpt:
  1.02 s: No fall predicted; score 0.106; margin -2.135
  1.27 s: No fall predicted; score 0.158; margin -1.671
fall known training excerpt:
  1.00 s: Suspected fall; score 0.731; margin 1.000
  1.28 s: Suspected fall; score 0.731; margin 1.000
